# Qwen3-VL-8B CSS Benchmark

This notebook generates ten CSS files from text specifications plus fixed HTML, renders them in Chromium at 800×600, and compares them with reference renders. Select an A100 GPU runtime before continuing.

In [ ]:
!nvidia-smi
!pip -q install "transformers>=4.57.0" "accelerate>=1.1.0" "bitsandbytes>=0.45.0" "qwen-vl-utils>=0.0.14" "playwright>=1.48.0" "Pillow>=10.4.0" "numpy>=2.0.0"
!playwright install chromium

Upload `qwen3-vl-css-benchmark.zip` when prompted if the benchmark folder is not already present under `/content`.

In [ ]:
from pathlib import Path
import zipfile

candidates = [Path('/content/qwen3-vl-css-benchmark'), Path.cwd()]
benchmark_root = next((p for p in candidates if (p / 'cases.json').exists()), None)

if benchmark_root is None:
    from google.colab import files
    uploaded = files.upload()
    archive_name = next(name for name in uploaded if name.endswith('.zip'))
    with zipfile.ZipFile(archive_name) as archive:
        archive.extractall('/content')
    benchmark_root = Path('/content/qwen3-vl-css-benchmark')

assert (benchmark_root / 'cases.json').exists(), 'Benchmark files were not found.'
print('Benchmark root:', benchmark_root)

In [ ]:
# BF16 fits on an A100. Add --load-in-4bit to the command for a smaller GPU.
import subprocess

command = [
    'python', str(benchmark_root / 'run_benchmark.py'),
    '--model', 'Qwen/Qwen3-VL-8B-Instruct',
    '--limit', '10'
]
subprocess.run(command, check=True)

In [ ]:
import json
import pandas as pd

scores = json.loads((benchmark_root / 'results/scores.json').read_text())
print('Mean score:', scores['mean_score'])
print('Overflow failures:', scores['overflow_failures'])
pd.DataFrame(scores['cases'])

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

rows = len(scores['cases'])
fig, axes = plt.subplots(rows, 2, figsize=(12, 4 * rows))
for row, case in enumerate(scores['cases']):
    case_id = case['id']
    reference = Image.open(benchmark_root / f'results/screenshots/{case_id}-reference.png')
    candidate = Image.open(benchmark_root / f'results/screenshots/{case_id}-candidate.png')
    axes[row, 0].imshow(reference)
    axes[row, 0].set_title(f'{case_id}: reference')
    axes[row, 1].imshow(candidate)
    axes[row, 1].set_title(f'candidate — {case["score"]:.4f}')
    axes[row, 0].axis('off')
    axes[row, 1].axis('off')
plt.tight_layout()

In [ ]:
# Download all generated CSS, screenshots, and scores.
import shutil
from google.colab import files

archive = shutil.make_archive('/content/qwen3-vl-css-results', 'zip', benchmark_root / 'results')
files.download(archive)